In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer

from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering

from sklearn.metrics import silhouette_score
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import calinski_harabasz_score

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from scipy.cluster.hierarchy import dendrogram, linkage



In [5]:
df = pd.read_csv("store.csv")
print("Dataset loaded Succesfully")
print("Shape of dataset :",df.shape)
df.head()

Dataset loaded Succesfully
Shape of dataset : (10000, 5)


,reps,product,qty,revenue,region
0,Chitra,Galaxy,2,155.10,West
1,Vijay,Jet,2,39.30,North
2,Mala,Beacon,3,74.25,West
3,Suman,Alpen,3,100.98,North
4,Rachna,Orbit,2,44.98,North


In [6]:
print("Number of rows :",df.shape[0])
print("Number of columns :",df.shape[1])
print("\nColumn names\n")
print(df.columns)
print("\nData Types\n")
print(df.dtypes)

Number of rows : 10000
Number of columns : 5

Column names

Index(['reps', 'product', 'qty', 'revenue', 'region'], dtype='str')

Data Types

reps           str
product        str
qty          int64
revenue    float64
region         str
dtype: object


In [7]:
print("Missing Values in each columns: ")
print(df.isnull().sum())

Missing Values in each columns: 
reps       0
product    0
qty        0
revenue    0
region     0
dtype: int64


In [8]:
print("Stastical Summary")
df.describe()

Stastical Summary


,qty,revenue
count,10000.000000,10000.000000
mean,3.387400,90.570221
std,4.318308,127.803928
min,1.000000,18.430000
25%,2.000000,39.300000
50%,2.000000,58.425000
75%,3.000000,75.000000
max,25.000000,1998.750000


In [9]:
print("Unique values in categorical columns: ")
print("\nReps:")
print(df["reps"].unique())
print("\nProduct:")
print(df["product"].unique())
print("\nRegion:")
print(df["region"].unique())

Unique values in categorical columns: 

Reps:
<ArrowStringArray>
[ 'Chitra',   'Vijay',    'Mala',   'Suman',  'Rachna',    'Aash',   'Chand',
   'Suraj',    'Bala',    'Seet',    'Ravi',  'Anusha',  'Ananya',   'Jaggi',
    'Reva',   'Meena',     'Sai',   'Rahul',     'Ram',  'Prarth',   'Pooja',
   'Rishi',   'Ranga',    'Sesh', 'Shaanth', 'Bharath',   'Susan',   'Ratna',
  'Easwar',  'Prasad',   'Madhu',  'Suresh',  'Anahit',  'Sumedh',   'Manju',
 'Nandini', 'Santosh',  'Satyen',  'Vaghya',   'Palak',  'Aparna', 'Jagdish',
    'Vish',   'Kumar',    'John',    'Bhat', 'Hussain',    'Alka',  'Rohini',
  'Keshab',   'Nidhi',   'Satya',  'Venkat',  'Veeyes',   'Rajat', 'Chandra',
   'Kamat',    'Raji',   'Swami',     'Jay',  'Mukund',   'Priya',  'Kishen',
   'Vidya',   'Akila',   'Durga',     'Mak',   'Mehta',   'Javed',   'Sruti',
  'Mukesh', 'Kishore']
Length: 72, dtype: str

Product:
<ArrowStringArray>
[ 'Galaxy',     'Jet',  'Beacon',   'Alpen',   'Orbit', 'Trident',    'Mars',
  

<h2>4.Data Cleaning and Feature Selection</h2>

The clustering features are `qty`, `revenue`, `reps`, `product` and `region`. Missing numeric values are filled with the median and missing categorical values with the most frequent value. Duplicate rows are checked and removed.

In [10]:
# Work on a copy so the raw dataframe remains available for reference
df = df.copy()

print("Duplicate rows before cleaning:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

numerical_features = ["qty", "revenue"]
categorical_features = ["reps", "product", "region"]
required_features = numerical_features + categorical_features

missing_columns = [col for col in required_features if col not in df.columns]
if missing_columns:
    raise ValueError(f"Expected columns not found in store.csv: {missing_columns}")

print("Missing values before imputation:")
display(df[required_features].isnull().sum())

# Impute missing values with a simple, documented strategy
for col in numerical_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

for col in categorical_features:
    if df[col].isnull().any():
        mode_values = df[col].mode(dropna=True)
        fill_value = mode_values.iloc[0] if not mode_values.empty else "Unknown"
        df[col] = df[col].fillna(fill_value)

print("Missing values after imputation:")
display(df[required_features].isnull().sum())
print("Cleaned dataset shape:", df.shape)

Duplicate rows before cleaning: 3252
Shape after removing duplicates: (6748, 5)
Missing values before imputation:


qty        0
revenue    0
reps       0
product    0
region     0
dtype: int64

Missing values after imputation:


qty        0
revenue    0
reps       0
product    0
region     0
dtype: int64

Cleaned dataset shape: (6748, 5)


<h2>5.Prepeocessing </h2>

Numerical features are standardised so that `revenue` does not dominate `qty` in the distance calculations. Categorical features are one-hot encoded because the clustering algorithms need numeric input.
Since clustering has no target variable and no train/test split, the transformer is fitted on the cleaned data used for clustering.

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numerical_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

X = preprocessor.fit_transform(df[required_features])
print("Preprocessing completed")
print("Shape of preprocessed data:", X.shape)

Preprocessing completed
Shape of preprocessed data: (6748, 90)
